In [3]:
# check you are in the right environment
import sys
import yt_dlp
print(f"Python Version: {sys.version.split()[0]}")
print(f"yt-dlp Version: {yt_dlp.version.__version__}")

Python Version: 3.10.20
yt-dlp Version: 2026.03.17


In [13]:
# if bookmarked in fb

import re

# 1. READ RAW LINKS
with open("links.txt", "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

converted_links = []
watch_count = 0

print(f"Read {len(raw_lines)} lines from links.txt")

# 2. CONVERSION LOGIC
for line in raw_lines:
    url = line.strip()
    
    if "facebook.com/watch/" in url:
        # We use a Regular Expression to find the numbers after 'v='
        match = re.search(r'v=(\d+)', url)
        if match:
            video_id = match.group(1)
            new_url = f"https://www.facebook.com/reel/{video_id}"
            converted_links.append(new_url)
            watch_count += 1
        else:
            # If it's a watch link but has no ID, we just keep it to let Cell 1 handle it
            converted_links.append(url)
    else:
        # Keep reels and other links as they are
        converted_links.append(url)

# 3. OVERWRITE LINKS.TXT WITH CLEANED/CONVERTED LINKS
with open("links.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(converted_links))

print(f"✨ Success!")
print(f"🔄 Converted {watch_count} 'Watch' links into 'Reel' format.")
print(f"📝 links.txt has been updated and is ready for Cell 1.")

Read 31 lines from links.txt
✨ Success!
🔄 Converted 16 'Watch' links into 'Reel' format.
📝 links.txt has been updated and is ready for Cell 1.


In [14]:
# 1. READ RAW LINKS
with open("links.txt", "r", encoding="utf-8") as f:
    raw_links = f.readlines()

print(f"📊 Total raw links found in links.txt: {len(raw_links)}")
print("First 3 raw links:")
for l in raw_links[:3]:
    print(f"  - {l.strip()}")

# 2. FILTER FOR FACEBOOK REELS AND WATCH VIDEOS
video_links = []
junk_count = 0

for link in raw_links:
    url = link.strip()
    
    # Check for reels with IDs (must have characters after /reel/)
    # We split by '/reel/' and check if the second part is longer than 5 chars
    is_reel = "/reel/" in url and len(url.split("/reel/")[-1].split('?')[0]) > 5
    
    # Check for watch links with IDs (must have characters after /watch/)
    is_watch = "/watch/" in url and len(url.split("/watch/")[-1].split('?')[0]) > 5
    
    if is_reel or is_watch:
        # Strip tracking junk (everything after ?)
        clean_link = url.split('?')[0].rstrip('/')
        
        # Avoid duplicates
        if clean_link not in video_links:
            video_links.append(clean_link)
    else:
        # This counts things like 'facebook.com/watch/' with no ID
        junk_count += 1

# 3. SAVE FILTERED LINKS
with open("video_links.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(video_links))

# 4. FINAL OUTPUT
print("\n" + "="*30)
print(f"✅ Extraction Complete!")
print(f"🗑️ Junk links ignored:      {junk_count}")
print(f"🎥 Total Facebook videos found: {len(video_links)}")
print("="*30)
print(f"📁 Saved to: video_links.txt")
print("Preview of filtered links:")
for l in video_links[:2]:
    print(f"  - {l}")

📊 Total raw links found in links.txt: 31
First 3 raw links:
  - https://www.facebook.com/
  - https://www.facebook.com/friends/
  - https://www.facebook.com/groups/

✅ Extraction Complete!
🗑️ Junk links ignored:      15
🎥 Total Facebook videos found: 16
📁 Saved to: video_links.txt
Preview of filtered links:
  - https://www.facebook.com/reel/27853165114309278
  - https://www.facebook.com/reel/1919445062006064


In [15]:
import yt_dlp
import os
from tqdm import tqdm
import time

# --- CONFIGURATION ---
BATCH_ID = "FB_002"
BASE_DIR = os.path.join(os.path.expanduser("~"), "Videos", "fb_downloads", BATCH_ID)
HISTORY_FILE = "fb_history.txt"
INPUT_FILE = "video_links.txt"

# Ensure folders exist
os.makedirs(BASE_DIR, exist_ok=True)

# --- CLEAN LOGGER (Hides the messy text) ---
class MyLogger:
    def debug(self, msg): pass
    def warning(self, msg): pass
    def error(self, msg):
        # We only want to see actual errors
        if "re-login" in msg.lower() or "private" in msg.lower():
            print(f"\n❌ {msg}")

# --- DOWNLOADER FUNCTION ---
def download_batch():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ Error: {INPUT_FILE} not found!")
        return

    with open(INPUT_FILE, "r") as f:
        links = [line.strip() for line in f if line.strip()]

    total_videos = len(links)
    success_count = 0
    failed_count = 0
    skipped_count = 0
    
    print(f"🚀 Starting Facebook Batch {BATCH_ID}")
    print(f"📂 Saving to: {BASE_DIR}\n")

    # The Progress Bar
    pbar = tqdm(total=total_videos, desc="Progress", unit="vid", colour='green', dynamic_ncols=True)

    ydl_opts = {
        "format": "bestvideo+bestaudio/best",
        "merge_output_format": "mp4",
        "quiet": False,             # Hides standard output
        "no_warnings": True,       # Hides warnings
        "logger": MyLogger(),      # Uses our custom silent logger
        "cookiesfrombrowser": ("firefox",),
        "download_archive": HISTORY_FILE,
        "outtmpl": os.path.join(BASE_DIR, "%(uploader)s", "%(id)s.%(ext)s"),
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        for i, link in enumerate(links, 1):
            # Update status message on the bar
            remaining = total_videos - i
            pbar.set_postfix_str(f"Done: {success_count} | Left: {remaining}")
            
            try:
                # We check the archive manually to count "skips"
                info = ydl.extract_info(link, download=False)
                # If we reach here, the video is valid. Let's download:
                result = ydl.download([link])
                
                # yt-dlp returns 0 for success, but it returns 0 even if skipped via archive.
                # So we just increment success based on logic.
                success_count += 1
                
            except Exception:
                failed_count += 1
                with open("fb_failed.txt", "a") as f_err:
                    f_err.write(f"{link}\n")
            
            pbar.update(1)

    pbar.close()

    # --- FINAL SUMMARY ---
    print("\n" + "="*40)
    print("🏁 DOWNLOAD SESSION COMPLETE")
    print(f"✅ Successfully Downloaded: {success_count}")
    print(f"❌ Failed/Blocked:        {failed_count}")
    print(f"📊 Total Processed:       {total_videos}")
    print("="*40)

# Run the downloader
download_batch()

🚀 Starting Facebook Batch FB_002
📂 Saving to: C:\Users\HP\Videos\fb_downloads\FB_002



Progress: 100%|██████████| 16/16 [01:53<00:00,  7.07s/vid, Done: 15 | Left: 0]


🏁 DOWNLOAD SESSION COMPLETE
✅ Successfully Downloaded: 16
❌ Failed/Blocked:        0
📊 Total Processed:       16
